# 2_2_05 — Silver: lead activities root `to` child table

Explodes the root-level `to` array of `silver.lead_activites_processed` into `silver.lead_activites_processed_to` — one row per direct recipient, keyed to the activity's `lead_id`.

Note (verified Nov 1 2026): all `to` values are PII-masked (`user_XXXXXX@masked.com`); 56% of distinct values match CRM contacts by exact masked email, extending the masked-email identity bridge.

In [0]:
%run ../utils

In [0]:
# 2_2_05: root-level 'to' array child table - one row per direct recipient
# Option B CDC (parent-driven, engine utils): watermark on THIS child (bronze_updated_at
# carried down), changed parents only, LATEST version per activity_id, exact DELETE of
# affected activity_ids, then engine write_silver. First run / legacy schema = full rebuild.
from pyspark.sql.functions import col, explode_outer, row_number
from pyspark.sql.window import Window

CHILD = 'lead_activites_processed_to'
try:
    watermark = get_silver_watermark(CHILD)
except Exception:
    watermark = None

src = spark.table('crm_ingestion.silver.lead_activites_processed').select(
    'id', 'lead_id', 'bronze_insert_date', 'bronze_updated_at', 'to')
if watermark is not None:
    wm_str = watermark.strftime('%Y-%m-%d %H:%M:%S.%f')
    print(f"CDC MODE: parents updated after {wm_str}")
    src = src.filter(col('bronze_updated_at') > watermark)
else:
    print("FULL LOAD: all activities, latest version per activity_id")

_version = Window.partitionBy('id').orderBy(col('bronze_updated_at').desc(), col('bronze_insert_date').desc())
df_in = src.withColumn('_v', row_number().over(_version)).filter(col('_v') == 1).drop('_v')

if watermark is None or len(df_in.take(1)) > 0:
    if watermark is not None:
        spark.sql(f"""
            DELETE FROM crm_ingestion.silver.{CHILD}
            WHERE activity_id IN (
                SELECT id FROM crm_ingestion.silver.lead_activites_processed
                WHERE bronze_updated_at > '{wm_str}'
            )""")
        _CDC_WATERMARK = watermark
    df_to = (
        df_in.select(
            col('id').alias('activity_id'),
            'lead_id', 'bronze_insert_date', 'bronze_updated_at',
            explode_outer(col('to')).alias('to_email')
        )
        .dropna(subset=['to_email'])
    )
    write_silver(df_to, CHILD)
    _CDC_WATERMARK = None
else:
    print("  No changed activities — nothing to do")